# Advanced Sparse Coding: Image Denoising with Dictionary Learning

In this lab you will apply sparse coding and dictionary learning to a **real-world task: image denoising**.

You will:
1. Add synthetic Gaussian noise to an image
2. Learn an overcomplete dictionary from the **noisy** patches
3. Denoise the image via sparse reconstruction
4. Evaluate quality with PSNR and SSIM
5. Compare sparse-coding denoising with PCA-based denoising
6. Visualize learned dictionary atoms and analyze sparsity trade-offs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, color, transform, util
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from sklearn.decomposition import MiniBatchDictionaryLearning, PCA
from sklearn.feature_extraction.image import extract_patches_2d, reconstruct_from_patches_2d

## Step 1 — Load & corrupt the image

Load the astronaut image, convert to grayscale, resize to half, and add Gaussian noise with $\sigma = 0.08$.

In [ ]:
# Load and prepare the clean image
face = data.astronaut()
face_gray = color.rgb2gray(face)
H, W = face_gray.shape
clean = transform.resize(face_gray, (H // 2, W // 2), anti_aliasing=True)

# Add Gaussian noise (sigma = 0.08)
sigma_noise = 0.08
noisy = clean + sigma_noise * np.random.randn(*clean.shape)
noisy = np.clip(noisy, 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(clean, cmap='gray'); axes[0].set_title('Clean'); axes[0].axis('off')
axes[1].imshow(noisy, cmap='gray'); axes[1].set_title(f'Noisy (σ={sigma_noise})'); axes[1].axis('off')
plt.tight_layout(); plt.show()
print(f'Noisy image PSNR: {psnr(clean, noisy):.2f} dB')

## Step 2 — Extract & normalize patches from the **noisy** image

Extract overlapping $7 \times 7$ patches, normalize each patch to zero-mean and unit-norm.

**Why 7×7?** Larger patches capture more structure but increase dictionary size. This is a good trade-off for denoising.

In [ ]:
pH, pW = 7, 7
n_patches = 50000

# Extract patches from the NOISY image
patches = extract_patches_2d(noisy, (pH, pW), max_patches=n_patches, random_state=42)
print(f'Extracted {patches.shape[0]} patches of size {patches.shape[1:]}')

# Reshape to 2D matrix: (n_patches, pH*pW)
X = patches.reshape(patches.shape[0], -1).astype(np.float64)

# Store per-patch mean for later reconstruction
patch_means = X.mean(axis=1, keepdims=True)

# Center each patch (subtract its own mean)
X -= patch_means

# Normalize by global standard deviation
global_std = X.std()
X /= global_std

print(f'Training matrix shape: {X.shape}')

## Step 3 — Learn an overcomplete dictionary

Use `MiniBatchDictionaryLearning` to learn an overcomplete dictionary with **128 atoms** (the patch dimension is $7 \times 7 = 49$, so 128 > 49 makes it overcomplete).

**Task:** Create the dictionary learner and fit it. Experiment with `alpha` (regularization strength). A higher `alpha` enforces more sparsity.

In [ ]:
n_atoms = 128

# YOUR CODE HERE: create MiniBatchDictionaryLearning with
#   n_components=n_atoms, alpha=1.5, max_iter=800,
#   transform_algorithm='omp', transform_n_nonzero_coefs=4,
#   random_state=42, n_jobs=-1
dl = ### YOUR CODE HERE ###

# Fit the dictionary on the normalized noisy patches
### YOUR CODE HERE ###

D = dl.components_
print(f'Dictionary shape: {D.shape}  (n_atoms × patch_dim)')

## Step 4 — Visualize the learned dictionary atoms

Display all 128 atoms as small images. Good atoms should look like oriented edges and textures.

In [ ]:
fig, axes = plt.subplots(8, 16, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    if i < D.shape[0]:
        ax.imshow(D[i].reshape(pH, pW), cmap='gray', interpolation='nearest')
    ax.axis('off')
fig.suptitle(f'Learned Dictionary — {n_atoms} atoms of size {pH}×{pW}', fontsize=14)
plt.tight_layout(); plt.show()

## Step 5 — Denoise the image via sparse reconstruction

For denoising we use **all overlapping patches** of the noisy image, not a random subset.

**Pipeline:**
1. Extract all overlapping patches from the noisy image
2. Center and normalize them (same transform as training)
3. Obtain sparse codes with the learned dictionary
4. Reconstruct patches: $\hat{p} = \alpha D$
5. Un-normalize and add back the per-patch mean
6. Average overlapping patches to form the final image

Use `reconstruct_from_patches_2d` to handle the overlapping averaging.

In [ ]:
H_r, W_r = clean.shape  # resized image dimensions

# Extract ALL overlapping patches from the noisy image
all_patches = ### YOUR CODE HERE ###
print(f'Total overlapping patches: {all_patches.shape[0]}')

# Reshape, center, normalize
P = ### YOUR CODE HERE ###

# Sparse coding: find sparse representation of each patch
codes = ### YOUR CODE HERE ###

# Reconstruct patches: multiply codes by dictionary
recon = ### YOUR CODE HERE ###

# Un-normalize and add back per-patch mean
recon *= global_std
recon += p_means

# Reshape patches back to (n_patches, pH, pW)
recon_patches = recon.reshape(-1, pH, pW)

# Reconstruct full image by averaging overlapping patches
denoised_sc = ### YOUR CODE HERE ###
denoised_sc = np.clip(denoised_sc, 0, 1)

print(f'Denoised PSNR: {psnr(clean, denoised_sc):.2f} dB')
print(f'Denoised SSIM: {ssim(clean, denoised_sc, data_range=1.0):.4f}')

## Step 6 — Sparsity sweep

**Task:** Vary the number of non-zero coefficients from 1 to 12 and plot PSNR vs. sparsity. What is the optimal number of coefficients?

In [ ]:
coefs_range = range(1, 13)
psnr_values = []
ssim_values = []

for n_coefs in coefs_range:
    dl.set_params(transform_algorithm='omp', transform_n_nonzero_coefs=n_coefs)
    
    # YOUR CODE HERE: compute sparse codes, reconstruct, un-normalize, rebuild image
    codes_k = ### YOUR CODE HERE ###
    recon_k = ### YOUR CODE HERE ###
    recon_k *= global_std
    recon_k += p_means
    recon_patches_k = recon_k.reshape(-1, pH, pW)
    img_k = np.clip(reconstruct_from_patches_2d(recon_patches_k, (H_r, W_r)), 0, 1)
    
    psnr_values.append(psnr(clean, img_k))
    ssim_values.append(ssim(clean, img_k, data_range=1.0))
    print(f'  n_coefs={n_coefs:2d}  PSNR={psnr_values[-1]:.2f}  SSIM={ssim_values[-1]:.4f}')

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(list(coefs_range), psnr_values, 'b-o', label='PSNR')
ax1.set_xlabel('Number of non-zero coefficients')
ax1.set_ylabel('PSNR (dB)', color='b')
ax2 = ax1.twinx()
ax2.plot(list(coefs_range), ssim_values, 'r-s', label='SSIM')
ax2.set_ylabel('SSIM', color='r')
plt.title('Denoising quality vs. sparsity level')
fig.tight_layout(); plt.show()

## Step 7 — Compare with PCA denoising

PCA can also denoise by projecting onto the top-$k$ principal components (low-rank approximation). This discards noisy directions.

**Task:** Implement PCA-based denoising and compare PSNR with the sparse-coding approach.

In [ ]:
# PCA-based denoising
# YOUR CODE HERE:
# 1. Fit PCA on the normalized patches P with n_components = 20
# 2. Transform P to the PCA space 
# 3. Inverse-transform back to patch space 
# 4. Un-normalize and reconstruct the image

pca = ### YOUR CODE HERE ###
### YOUR CODE HERE ###

P_pca = ### YOUR CODE HERE ###
P_recon_pca = ### YOUR CODE HERE ###

P_recon_pca *= global_std
P_recon_pca += p_means
recon_patches_pca = P_recon_pca.reshape(-1, pH, pW)
denoised_pca = np.clip(reconstruct_from_patches_2d(recon_patches_pca, (H_r, W_r)), 0, 1)

psnr_pca = psnr(clean, denoised_pca)
ssim_pca = ssim(clean, denoised_pca, data_range=1.0)
print(f'PCA Denoising — PSNR: {psnr_pca:.2f} dB,  SSIM: {ssim_pca:.4f}')

In [ ]:
# Final comparison plot
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
imgs   = [clean, noisy, denoised_sc, denoised_pca]
titles = ['Clean',
          f'Noisy\nPSNR={psnr(clean,noisy):.1f}',
          f'Sparse Coding\nPSNR={psnr(clean,denoised_sc):.1f}',
          f'PCA (k=20)\nPSNR={psnr_pca:.1f}']
for ax, im, t in zip(axes, imgs, titles):
    ax.imshow(im, cmap='gray', vmin=0, vmax=1)
    ax.set_title(t); ax.axis('off')
plt.tight_layout(); plt.show()

## Questions to answer

1. Why does an **overcomplete** dictionary (128 atoms for 49-dimensional patches) help denoising?
2. What happens when you increase `alpha`? When you decrease it?
3. At what sparsity level do you observe the best PSNR? Why does going beyond that hurt?
4. Why does sparse coding outperform PCA for denoising? In which scenario might PCA be preferred?
5. **Bonus:** Try different noise levels ($\sigma = 0.05, 0.15, 0.25$) — how does the optimal number of coefficients change?